# Anandi Park — Generate TTS Audio with IndicF5

**Before you start:**
1. Go to https://huggingface.co/ai4bharat/IndicF5 and click **Agree and access**
2. Make sure runtime is GPU: Runtime > Change runtime type > T4 GPU
3. Run each cell top to bottom

After all cells run, you'll have 4 WAV files ready to download.

In [ ]:
# Cell 1: Install everything
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q soundfile numpy huggingface_hub
print("\n--- Install done. Now RESTART THE RUNTIME ---")
print("Click: Runtime > Restart session")
print("Then skip this cell and run Cell 2 onwards.")

In [ ]:
# Cell 2: Login to HuggingFace (run AFTER restart)
# The model is gated, so you need to authenticate.
# Replace YOUR_TOKEN_HERE with your token from:
# https://huggingface.co/settings/tokens
import os
os.environ["HF_TOKEN"] = "YOUR_TOKEN_HERE"  # <-- paste your token here

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("Logged in to HuggingFace.")

In [ ]:
# Cell 3: Load the model (~3GB download on first run)
from transformers import AutoModel
import numpy as np
import soundfile as sf
import os

print("Loading IndicF5 model...")
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)
print("Model loaded!")

In [ ]:
# Cell 4: Define call scripts

SCRIPTS = {
    "pitch-hindi": {
        "text": (
            "नमस्ते! मैं Anandi Park से बोल रहा हूँ। "
            "वाघोली-बकोरी रोड, पुणे पर प्रीमियम NA प्लॉट्स उपलब्ध हैं। "
            "कीमत सिर्फ पंद्रह लाख से शुरू। RERA रजिस्टर्ड, क्लीयर टाइटल। "
            "चौरासी प्लॉट्स में से कुछ ही बचे हैं। "
            "अगर आपको साइट विज़िट करना है तो कृपया एक दबाइए। धन्यवाद!"
        ),
    },
    "pitch-marathi": {
        "text": (
            "नमस्कार! मी Anandi Park कडून बोलतोय। "
            "वाघोळी-बकोरी रोड, पुणे येथे प्रीमियम NA प्लॉट्स उपलब्ध आहेत। "
            "किंमत फक्त पंधरा लाखापासून सुरू. RERA नोंदणीकृत, स्पष्ट मालकी हक्क। "
            "चौऱ्यांशी प्लॉट्सपैकी काही शिल्लक आहेत। "
            "साइट भेट करायची असल्यास कृपया एक दाबा. धन्यवाद!"
        ),
    },
    "pitch-english": {
        "text": (
            "Hello! I am calling from Anandi Park. "
            "Premium NA plots are available on Wagholi Bakori Road, Pune. "
            "Prices start from just fifteen lakhs. RERA registered, clear titles. "
            "Out of eighty four plots, only a few remain. "
            "If you would like to schedule a site visit, please press one. Thank you!"
        ),
    },
    "followup-hindi": {
        "text": (
            "नमस्ते! मैं Anandi Park की टीम से बोल रहा हूँ। "
            "कुछ दिन पहले आपने हमारे प्लॉट्स के बारे में जानकारी ली थी। "
            "इस हफ्ते फ्री साइट विज़िट का मौका है। "
            "अगर आप आना चाहते हैं तो एक दबाइए। "
            "हमारी टीम आपसे संपर्क करेगी। धन्यवाद!"
        ),
    },
}

print(f"Defined {len(SCRIPTS)} scripts:")
for name, s in SCRIPTS.items():
    print(f"  {name} — {len(s['text'])} chars")

In [ ]:
# Cell 5: Generate audio
import urllib.request

os.makedirs("output", exist_ok=True)

# Download reference audio for voice style
REF_AUDIO = "ref_prompt.wav"
REF_TEXT = "नमस्ते मेरा नाम गीता है क्या यह आपसे बात करने का सही समय है"

if not os.path.exists(REF_AUDIO):
    url = "https://huggingface.co/ai4bharat/IndicF5/resolve/main/prompts/HIN_F_HAPPY_00001.wav"
    headers = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(REF_AUDIO, 'wb') as f:
        f.write(resp.read())
    print(f"Downloaded reference audio")

results = {}

for name, script in SCRIPTS.items():
    print(f"\nGenerating: {name}...")
    try:
        audio = model(
            script["text"],
            ref_audio_path=REF_AUDIO,
            ref_text=REF_TEXT,
        )
        if isinstance(audio, np.ndarray):
            if audio.dtype == np.int16:
                audio = audio.astype(np.float32) / 32768.0
        else:
            audio = np.array(audio, dtype=np.float32)

        out_path = f"output/{name}.wav"
        sf.write(out_path, audio, samplerate=24000)
        duration = len(audio) / 24000
        results[name] = out_path
        print(f"  Done: {out_path} ({duration:.1f}s)")
    except Exception as e:
        print(f"  Failed: {e}")

print(f"\nGenerated {len(results)} audio files.")

In [ ]:
# Cell 6: Listen to previews
from IPython.display import Audio, display

for name, path in results.items():
    print(f"\n{name}:")
    display(Audio(path))

In [ ]:
# Cell 7: Download all files
from google.colab import files

for name, path in results.items():
    files.download(path)
    print(f"Downloaded: {path}")

print("\nDone! Upload these to your VPS:")
print("  scp output/*.wav root@147.93.169.183:/opt/anandi-park/anandi/uploads/tts/")